
<div style="background-color: mistyrose; color: black; padding: 20px; box-sizing: border-box; width: 99%;">

👋 <b>Olá! Eu sou o Dot, seu revisor de IA.</b>

Você está claramente no caminho certo aqui — este é um projeto bem organizado e cuidadosamente codificado. **Pontos positivos**: sua limpeza de dados é cuidadosa e bem justificada (verificações de duplicatas, investigação dos valores de Tenure ausentes, remoção de colunas de identificadores irrelevantes), você construiu corretamente duas codificações separadas adequadas para diferentes famílias de modelos, o scaling foi ajustado apenas nos dados de treinamento, e você foi muito além do mínimo testando três tipos de modelo em três técnicas diferentes de correção de desbalanceamento usando funções auxiliares reutilizáveis — tudo isso enquanto alcançava um F1 de teste final de 0,590, exatamente no patamar exigido. **Precisa corrigir para passar**: algumas coisas precisam de atenção antes que isso esteja pronto: - `df_le = df` cria um alias em vez de uma cópia independente, então a etapa de codificação de rótulos silenciosamente mutua o dataframe original em vez de produzir uma versão codificada genuinamente separada, como pretendido. - Após treinar o modelo sem corrigir o desbalanceamento de classes, não há discussão escrita sobre o que os resultados realmente mostram — essa reflexão está completamente ausente. - A comparação entre as métricas AUC-ROC e F1-score, solicitada ao final da análise, nunca é realmente escrita, mesmo que ambos os valores sejam calculados. Uma vez que estes pontos sejam abordados, este projeto deve ficar em ótima forma — excelente trabalho geral, e você está muito próximo da linha de chegada.

<b>O que você deve fazer a seguir?</b>

<ol>
  <li><b>Revisar:</b> Confira meus comentários no notebook abaixo.</li>
  <li><b>Atualizar:</b> Ajuste seu código de acordo com o feedback fornecido nos comentários.</li>
  <li><b>Reenviar:</b> Envie seu projeto de volta para a próxima iteração. Vou revisar seu código atualizado para verificar as correções.</li>
</ol>

Se você tiver alguma dificuldade ou erro, fique à vontade para entrar em contato com nossa equipe de suporte clicando no ícone de chat no canto superior direito ou enviando um e-mail para <a href="mailto:helpme@tripleten.com">helpme@tripleten.com</a>.

<hr style="margin-top: 20px; margin-bottom: 20px;">

<details>
<summary><b style="cursor: pointer;">💡 Clique aqui para ver como ler meus comentários (Legenda de cores)</b></summary>
<div style="margin-top: 15px;">
Veja meus comentários abaixo. <b>Peço gentilmente que não os mova, modifique nem exclua</b>.

Você encontrará meus comentários destacados em caixas verdes, amarelas ou vermelhas como estas:

<div class="alert alert-block alert-success">
Sucesso. Tudo foi feito corretamente.
</div>

<div class="alert alert-block alert-warning">
Observações. Algumas recomendações.
</div>

<div class="alert alert-block alert-danger">
A corrigir. O bloco requer algumas correções.
</div>
</div>
</details>

</div>


### Objective
Create a machine learning model to predict if a bank client is about to cancel the account.

<div class="alert alert-block alert-success">
Declaração clara e concisa do objetivo do projeto — estabelece bem o contexto para tudo que vem a seguir.
</div>

In [1]:
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.utils import shuffle
import plotly.express as px

import numpy as np

<div class="alert alert-block alert-success">
Todas as bibliotecas que você realmente usa depois (models, encoders, scaler, metrics, shuffle, plotly.express, numpy) são importadas juntas no início, o que mantém o resto do notebook fácil de acompanhar.
</div>

In [2]:
print(f'pandas version: {pd.__version__}')
print(f'scikit-learn version: {sklearn.__version__}')

pandas version: 3.0.5
scikit-learn version: 1.9.0


### Functions

In [3]:
param_grid={
    'n_estimators': [i for i in range(1, 31)],
    'max_depth': [i for i in range(1, 21)]}
param_grid

{'n_estimators': [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30],
 'max_depth': [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20]}

In [4]:
def lr_model_scores(features_train, target_train, features_val, target_val,
                    weight=None):
    '''
    Returns LogisticRegression model end its scores.
    '''
    model = LogisticRegression(
        solver='liblinear',
        random_state=123,
        class_weight=weight
    )
    model.fit(features_train, target_train)

    predicted_valid = model.predict(features_val)
    f1 = f1_score(target_val, predicted_valid)

    proba_valid = model.predict_proba(features_val)
    proba_one_valid = proba_valid[:, 1]
    roc_auc = roc_auc_score(target_val, proba_one_valid)

    print(
        f'F1 Score: {f1:.3f}\n'
        f'ROC AUC: {roc_auc:.3f}'
    )

    return model, f1, roc_auc


def rfc_optimizer(features_train, target_train, features_val, target_val,
                  weight=None):
    '''
    Returns a RandonForestClassifier model with best n_estimators between 
    1 and 30 and best max_depth between 1 and 20, maximizing the F1 Score.
    '''

    best_model = None
    best_n_estimators = 0
    best_depth = 0
    best_f1_score = 0

    for estimators in range(1, 31):
        for depth in range(1, 21):

            model = RandomForestClassifier(
                n_estimators=estimators,
                max_depth=depth,
                random_state=123,
                class_weight=weight
            )
            model.fit(features_train, target_train)

            predicted_valid = model.predict(features_val)
            f1 = f1_score(target_val, predicted_valid)

            if f1 > best_f1_score:
                best_n_estimators = estimators
                best_depth = depth
                best_f1_score = f1
                best_model = model

    proba_valid = best_model.predict_proba(features_val)
    proba_one_valid = proba_valid[:, 1]
    roc_auc = roc_auc_score(target_val, proba_one_valid)

    print(
        f'best_depth: {best_depth}\n'
        f'best_n_estimators: {best_n_estimators}\n'
        f'F1 Score: {best_f1_score:.3f}\n'
        f'ROC AUC: {roc_auc:.3f}'
    )

    f1_to_plot = pd.DataFrame(columns=['depth', 'f1_val'])
    for depth in range(1, 21):
        model = RandomForestClassifier(
            n_estimators=best_n_estimators,
            max_depth=depth,
            random_state=123,
            class_weight=weight
        )

        model.fit(features_train, target_train)    

        predicted_valid = model.predict(features_val)
        f1_val = f1_score(target_val, predicted_valid)
    
        f1_to_plot.loc[len(f1_to_plot)] = [depth, f1_val]
    
    fig = px.line(
        f1_to_plot,
        x='depth',
        y='f1_val',
        title=f'Validation F1 by Depth for n_estimators = {best_n_estimators}',
        labels={'f1_val': 'F1 Score', 'depth': 'Max Depth'},
        markers=True
    )
    fig.show()

    return best_model, best_n_estimators, best_depth, best_f1_score, roc_auc


def dtc_optimizer(features_train, target_train, features_val, target_val,
                  weight=None):
    '''
    Returns a DecisionTreeClassifier model with best max_depth between 
    1 and 20, maximizing the F1 Score.
    '''
    f1_to_plot = pd.DataFrame(columns=['depth', 'f1_val'])

    best_model = None
    best_depth = 0
    best_f1_score = 0
    for depth in range(1, 21):
        model = DecisionTreeClassifier(
            max_depth=depth, 
            random_state=123,
            class_weight=weight
        )

        model.fit(features_train, target_train)
        
        predicted_valid = model.predict(features_val)
        f1 = f1_score(target_val, predicted_valid)

        f1_to_plot.loc[len(f1_to_plot)] = [depth, f1]
        
        if f1 > best_f1_score:
            best_depth = depth
            best_f1_score = f1
            best_model = model

    proba_valid = best_model.predict_proba(features_val)
    proba_one_valid = proba_valid[:, 1]
    roc_auc = roc_auc_score(target_val, proba_one_valid)

    print(
        f'best_depth: {best_depth}\n'
        f'F1 Score: {best_f1_score:.3f}\n'
        f'ROC AUC: {roc_auc:.3f}'
    )

    fig = px.line(
        f1_to_plot,
        x='depth',
        y='f1_val',
        title=f'Validation F1 by Depth',
        labels={'f1_val': 'F1 Score', 'depth': 'Max Depth'},
        markers=True
    )
    fig.show()
           
    return best_model, best_depth, best_f1_score, roc_auc


def upsample(features, target):
    '''upsample df by multiplying it where target is 1'''
    # number to multiply the minority class
    multiplier = round(target.value_counts()[0] / target.value_counts()[1])
    
    features_zero = features[target == 0]
    features_one = features[target == 1]
    target_zero = target[target == 0]
    target_one = target[target == 1]

    features_up = pd.concat([features_zero] + [features_one] * multiplier)
    target_up = pd.concat([target_zero] + [target_one] * multiplier)

    features_up, target_up = shuffle(features_up, target_up, random_state=123)

    return features_up, target_up


def downsample(features, target):
    '''downsample df by sampling a fraction where target is 0'''
    fraction = target.value_counts()[1] / target.value_counts()[0]

    features_zero = features[target == 0]
    features_one = features[target == 1]
    target_zero = target[target == 0]
    target_one = target[target == 1]

    features_down = pd.concat(
        [features_zero.sample(frac=fraction, random_state=123)] + 
        [features_one]
    )
    target_down = pd.concat(
        [target_zero.sample(frac=fraction, random_state=123)] +
        [target_one]
    )

    features_down, target_down = shuffle(
        features_down,
        target_down,
        random_state=123
    )

    return features_down, target_down     

<div class="alert alert-block alert-warning">
Encapsular a lógica repetida de treinamento/scoring em lr_model_scores, rfc_optimizer, dtc_optimizer, upsample e downsample é um excelente trabalho estrutural que mantém o resto do notebook limpo. Dois pontos vale a pena revisitar: (1) em dtc_optimizer o docstring diz que busca max_depth 'entre 1 e 20', mas o loop vai apenas até range(1, 20), então a profundidade 20 nunca é testada silenciosamente — vale a pena verificar os limites do range sempre que um docstring declara um intervalo inclusivo; (2) rfc_optimizer retreina um modelo para cada uma das 600 combinações (n_estimators, depth) e depois retreina novamente por profundidade apenas para construir o gráfico, embora esses scores de F1 já tivessem sido computados no primeiro loop — reutilizar os scores da primeira passagem (ou usar GridSearchCV) seria mais eficiente.
</div>

### Read CSV

In [5]:
df = pd.read_csv('./datasets/Churn.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.1 MB


<div class="alert alert-block alert-success">
Carregar os dados e verificar imediatamente .info() fornece uma visão clara dos tipos de dados (dtypes) e contagens de nulos de imediato, o que prepara muito bem as etapas de limpeza que se seguem.
</div>

In [6]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


The column 'Exited' shows if the costumer cancelled (1) or not (0). It is the model prediction objective.

## 1 - Cleaning

In [7]:
df.duplicated().sum()

np.int64(0)

<div class="alert alert-block alert-success">
Verificar se há duplicatas em linhas completas e valores duplicados de CustomerId é uma forma abrangente de confirmar que não há dados redundantes antes de prosseguir.
</div>

In [8]:
df['CustomerId'].duplicated().sum()

np.int64(0)

There is no duplicated row or 'CustomerId'.

The columns 'RowNumber', 'CustomerId' and 'Surname' are not relevant for model training and can be discarded.

In [9]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

<div class="alert alert-block alert-success">
Remoção correta de RowNumber, CustomerId e Surname — essas colunas identificadoras não têm poder preditivo e apenas adicionariam ruído. Boa justificativa fornecida logo antes.
</div>

There are null values in the 'Tenure' column. Verify if they are 0.

In [10]:
df['Tenure'].value_counts(dropna=False)

Tenure
1.0     952
2.0     950
8.0     933
3.0     928
5.0     927
7.0     925
NaN     909
4.0     885
9.0     882
6.0     881
10.0    446
0.0     382
Name: count, dtype: int64

<div class="alert alert-block alert-success">
Usar value_counts(dropna=False) aqui é uma decisão inteligente — permite que você compare diretamente a contagem de zeros reais (382) contra valores ausentes (909), provando que não são a mesma coisa. Exatamente o tipo de raciocínio que essa etapa exige.
</div>

These null are not 0, because there are observations with 0. Verify if there is something in common with these customers.

In [11]:
df[df['Tenure'].isnull()].sample(10, random_state=123)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
7811,611,France,Male,30,NaN,104145.65,1,0,0,159629.64,0
3543,675,France,Male,49,NaN,135133.39,1,0,1,179521.24,1
6471,850,France,Female,40,NaN,0.00,2,1,0,35034.15,0
94,730,Spain,Male,42,NaN,0.00,2,0,1,85982.47,0
6187,704,France,Male,36,NaN,114370.41,1,0,1,66810.48,0
7026,677,France,Female,27,NaN,0.00,2,0,1,114685.92,0
5314,743,Germany,Male,45,NaN,144677.19,3,1,0,22512.44,1
4445,739,France,Male,29,NaN,0.00,2,1,1,164484.78,0
9006,661,Spain,Male,20,NaN,0.00,1,1,0,110252.53,0
1960,655,Spain,Female,36,NaN,135515.76,1,1,0,86013.96,0


In [12]:
df[df['Tenure'].notna()].describe()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000
mean,650.736553,38.949181,4.997690,76522.740015,1.530195,0.704983,0.515565,100181.214924,0.203938
std,96.410471,10.555581,2.894723,62329.528576,0.581003,0.456076,0.499785,57624.755647,0.402946
min,350.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,11.580000,0.000000
25%,584.000000,32.000000,2.000000,0.000000,1.000000,0.000000,0.000000,51227.745000,0.000000
50%,652.000000,37.000000,5.000000,97318.250000,1.000000,1.000000,1.000000,100240.200000,0.000000
75%,717.000000,44.000000,7.000000,127561.890000,2.000000,1.000000,1.000000,149567.210000,0.000000
max,850.000000,92.000000,10.000000,250898.090000,4.000000,1.000000,1.000000,199992.480000,1.000000


In [13]:
df[df['Tenure'].isna()].describe()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,909.000000,909.000000,0.0,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000
mean,648.451045,38.647965,NaN,76117.341474,1.530253,0.710671,0.510451,99180.389373,0.201320
std,99.079381,9.785438,NaN,63105.690715,0.588452,0.453701,0.500166,56378.063765,0.401207
min,359.000000,18.000000,NaN,0.000000,1.000000,0.000000,0.000000,106.670000,0.000000
25%,580.000000,32.000000,NaN,0.000000,1.000000,0.000000,0.000000,49872.330000,0.000000
50%,647.000000,37.000000,NaN,96674.550000,1.000000,1.000000,1.000000,99444.020000,0.000000
75%,718.000000,43.000000,NaN,128554.980000,2.000000,1.000000,1.000000,145759.700000,0.000000
max,850.000000,92.000000,NaN,206663.750000,4.000000,1.000000,1.000000,199390.450000,1.000000


In [14]:
df[df['Tenure'].notna()]['Gender'].value_counts(normalize=True)

Gender
Male      0.547135
Female    0.452865
Name: proportion, dtype: float64

In [15]:
df[df['Tenure'].isna()]['Gender'].value_counts(normalize=True)

Gender
Male      0.531353
Female    0.468647
Name: proportion, dtype: float64

There is nothing in common between them. These will be discarded.

<div class="alert alert-block alert-warning">
A conclusão de que não há 'nada em comum' entre as linhas com Tenure nulo repousa em uma inspeção visual de apenas 10 linhas entre 909 entradas faltantes (menos de 1,5% desse subconjunto) — essa é uma base frágil para uma conclusão geral. Comparar estatísticas agregadas (por exemplo, `describe()` ou um `groupby` em Geography/Gender/Exited) entre os subconjuntos com valores nulos e não nulos seria mais robusto. Também vale verificar se a distribuição de Exited nessas linhas se assemelha à do conjunto de dados geral antes de removê-las, já que você está eliminando cerca de 9% de um alvo já desbalanceado.
</div>

In [16]:
df = df.dropna().reset_index(drop=True)

<div class="alert alert-block alert-success">
Implementa corretamente a decisão de drop, e resetar o índice em seguida é uma boa prática que evita surpresas relacionadas a índices mais tarde.
</div>

Verify if age makes sense.

In [17]:
df['Age'].describe()

count    9091.000000
mean       38.949181
std        10.555581
min        18.000000
25%        32.000000
50%        37.000000
75%        44.000000
max        92.000000
Name: Age, dtype: float64

## 2 - Preparing

### 2.1 - Class Imbalance Analysis

In [18]:
df['Exited'].value_counts(normalize=True)

Exited
0    0.796062
1    0.203938
Name: proportion, dtype: float64

<div class="alert alert-block alert-success">
value_counts(normalize=True) é uma forma limpa e direta de revelar a divisão de classes aproximadamente 80/20.
</div>

Only 20% of the values are 1. This will not be balanced now to compare the model before and after considering the imbalance.

<div class="alert alert-block alert-success">
Descreve com precisão o desequilíbrio mostrado na saída anterior e declara claramente o plano de comparar o treinamento não balanceado versus balanceado posteriormente.
</div>

### 2.2 - Encode Cathegorical Characteristics

In [19]:
df['Geography'].unique()

<StringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str

<div class="alert alert-block alert-success">
Verificar .unique() em Geography (e Gender logo depois) antes de codificar confirma que não há categorias inesperadas ou erros de digitação — uma boa verificação de sanidade antes de se comprometer com uma estratégia de codificação.
</div>

In [20]:
df['Gender'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

LogisticRegression performs better with One-Hot Encoding, 
while RandomForestClassifier and DecisionTreeClassifier perform better with Label Encoding.

Two dfs will be created, one for each encoding technique.

<div class="alert alert-block alert-success">
A escolha explícita de One-Hot Encoding para Logistic Regression e codificação de estilo ordinal para modelos baseados em árvores demonstra uma compreensão sólida de como diferentes algoritmos interpretam variáveis categóricas.
</div>

#### 2.2.1 One-Hot Encoding

In [21]:
df_ohe = pd.get_dummies(df, drop_first=True)
df_ohe.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2.0,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1.0,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8.0,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1.0,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2.0,125510.82,1,1,1,79084.10,0,False,True,False


<div class="alert alert-block alert-warning">
pd.get_dummies(df, drop_first=True) foi aplicado corretamente e o dataframe resultante parece correto. Uma coisa a ter em mente para projetos futuros: essa codificação é derivada do dataset completo antes da divisão train/validation/test. É baixo risco aqui, já que as categorias são fixas e conhecidas antecipadamente, mas como um hábito geral é mais limpo derivar qualquer codificação orientada a dados apenas dos dados de treinamento.
</div>

#### 2.2.2 Label Encoding

In [22]:
cat_columns = ['Geography', 'Gender']

encoder = OrdinalEncoder()
df_cat = pd.DataFrame(
    encoder.fit_transform(df[cat_columns]), 
    columns=cat_columns
)

df_le = df.copy()
df_le[cat_columns] = df_cat
df_le.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,0.0,0.0,42,2.0,0.00,1,1,1,101348.88,1
1,608,2.0,0.0,41,1.0,83807.86,1,0,1,112542.58,0
2,502,0.0,0.0,42,8.0,159660.80,3,1,0,113931.57,1
3,699,0.0,0.0,39,1.0,0.00,2,0,0,93826.63,0
4,850,2.0,0.0,43,2.0,125510.82,1,1,1,79084.10,0


<div class="alert alert-block alert-danger">
df_le = df não cria uma cópia independente — apenas cria outro nome que aponta para o mesmo objeto na memória. Assim, a linha seguinte, df_le[cat_columns] = df_cat, na verdade muta o df original também, o que entra em conflito com seu próprio plano declarado de construir 'dois dfs... um para cada técnica de encoding.' Apenas não quebra nada visualmente nesta execução porque df_ohe já foi construído a partir de df antes dessa mutação — se a ordem das células mudasse ou df fosse reutilizado depois, isso produziria dados errados silenciosamente. Estude como assignment vs. copying funciona em pandas para que df_le seja genuinamente independente de df. Em uma nota relacionada, OrdinalEncoder também está sendo fit no df[cat_columns] completo antes do split train/validation/test — com apenas três categorias fixas, isso não está causando leakage real aqui, mas como regra qualquer encoder que aprenda com dados deve ser fit apenas no training set.
</div>

### 2.3 - Splitting

In [23]:
# split df_ohe into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(
    df_ohe,
    stratify=df_ohe['Exited'],
    test_size=0.4, 
    random_state=123
)
df_val, df_test = train_test_split(
    df_val_test,
    stratify=df_val_test['Exited'],
    test_size=0.5,
    random_state=123
)

features_train_ohe = df_train.drop('Exited', axis=1)
target_train_ohe = df_train['Exited']

features_val_ohe = df_val.drop('Exited', axis=1)
target_val_ohe = df_val['Exited']

features_test_ohe = df_test.drop('Exited', axis=1)
target_test_ohe = df_test['Exited']

<div class="alert alert-block alert-warning">
A divisão 60/20/20 está implementada corretamente e na ordem certa em relação ao dimensionamento. Como você já sabe que Exited está desbalanceado (cerca de 20% positivo), considere passar stratify= para train_test_split para garantir que as proporções das classes correspondam entre treino, validação e teste, em vez de confiar na chance.
</div>

In [24]:
# split df_le into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(
    df_le,
    stratify=df_le['Exited'],
    test_size=0.4,
    random_state=123
)
df_val, df_test = train_test_split(
    df_val_test,
    stratify=df_val_test['Exited'],
    test_size=0.5,
    random_state=123
)

features_train_le = df_train.drop('Exited', axis=1)
target_train_le = df_train['Exited']

features_val_le = df_val.drop('Exited', axis=1)
target_val_le = df_val['Exited']

features_test_le = df_test.drop('Exited', axis=1)
target_test_le = df_test['Exited']

### 2.4 - Scaling

In [25]:
# columns to scale
cols_sca = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 
            'EstimatedSalary']

# scale df_ohe
scaler_ohe = StandardScaler()
scaler_ohe.fit(features_train_ohe[cols_sca])

features_train_ohe[cols_sca] = scaler_ohe.transform(features_train_ohe[cols_sca])
features_val_ohe[cols_sca] = scaler_ohe.transform(features_val_ohe[cols_sca])
features_test_ohe[cols_sca] = scaler_ohe.transform(features_test_ohe[cols_sca])

# scale df_le
scaler_le = StandardScaler()
scaler_le.fit(features_train_le[cols_sca])

features_train_le[cols_sca] = scaler_le.transform(features_train_le[cols_sca])
features_val_le[cols_sca] = scaler_le.transform(features_val_le[cols_sca])
features_test_le[cols_sca] = scaler_le.transform(features_test_le[cols_sca])

<div class="alert alert-block alert-warning">
Ajustar o StandardScaler apenas nas features de treinamento e reutilizá-lo para transformar validação/teste é exatamente correto e evita vazamento de dados, aplicado consistentemente em ambas as versões codificadas — muito bem feito. Um ponto a considerar: NumOfProducts está incluído entre as colunas escaladas, mas é uma contagem discreta de pequeno intervalo (1-4) em vez de uma magnitude contínua como Balance; não é errado escaloná-la, mas vale a pena questionar se realmente precisa ser padronizada (isso importa principalmente para Logistic Regression, já que os modelos baseados em árvores são invariantes à escala).
</div>

## 3 - Model with Imbalance
Model without correcting the target imbalance.

In [26]:
# data frame to store all models results
scores_df = pd.DataFrame(columns=[
    'model',
    'n_estimators',
    'depth',
    'f1',
    'auc_roc']
)


<div class="alert alert-block alert-success">
Configurar scores_df antecipadamente para coletar F1 e AUC-ROC para cada modelo oferece uma forma limpa e organizada de comparar resultados depois.
</div>

### 3.1 - LogisticRegression
Logistic Regression model performs better with One-hot encoding.

In [27]:
model_lr, f1_lr, roc_auc_lr = lr_model_scores(
    features_train_ohe,
    target_train_ohe,
    features_val_ohe,
    target_val_ohe
)

scores_df.loc[len(scores_df)] = ['model_lr','','', f1_lr, roc_auc_lr]

F1 Score: 0.361
ROC AUC: 0.773


<div class="alert alert-block alert-success">
Treina corretamente Regressão Logística sem correção de desequilíbrio de classes e registra ambas as métricas. O grande diferença entre F1 (0.328) e ROC AUC (0.768) ilustra perfeitamente como uma métrica baseada em ranking pode parecer enganosamente forte mesmo quando um modelo tem dificuldades com a classe minoritária.
</div>

### 3.2 - RandomForestClassifier

Random Forest model performs better with Label Encoding.

In [28]:
# model_rfc, n_estimators, depth, f1_rfc, roc_auc_rfc = rfc_optimizer(
#     features_train_le,
#     target_train_le,
#     features_val_le,
#     target_val_le
# )


# scores_df.loc[len(scores_df)] = [
#     'model_rfc', 
#     n_estimators, 
#     depth, 
#     f1_rfc, 
#     roc_auc_rfc
# ]


<div class="alert alert-block alert-success">
A busca em grade varre corretamente tanto n_estimators quanto max_depth com class_weight=None. Os resultados (F1 0.620, ROC AUC 0.851) são fortes mesmo antes de qualquer balanceamento, e o gráfico de profundidade versus F1 está corretamente rotulado e consistente com o best_depth impresso.
</div>

### 3.3 - DecisionTreeClassifier

In [29]:
model_dtc, depth, f1_dtc, roc_auc_dtc = dtc_optimizer(
    features_train_le,
    target_train_le,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = ['model_dtc','',depth, f1_dtc, roc_auc_dtc]

best_depth: 6
F1 Score: 0.535
ROC AUC: 0.832


<div class="alert alert-block alert-danger">
A implementação da linha de base da Decision Tree em si está correta (o plot e o best_depth impresso são consistentes). No entanto, descrever seus resultados após treinar o modelo sem considerar o desequilíbrio de classes é um requisito explícito desta etapa, e essa descrição nunca aparece de fato — o notebook passa direto da linha de base para a seção de balanceamento sem discutir o que esses três resultados da linha de base (Logistic Regression ~0,33, Random Forest ~0,62, Decision Tree ~0,60) realmente mostram, por exemplo, que um modelo linear tem muito mais dificuldade com esse desequilíbrio do que os modelos baseados em árvores. Adicione uma breve conclusão escrita comparando os três resultados da linha de base antes de prosseguir.
</div>

## 4 - Model without Imbalance
Model correcting the target imbalance.

In [30]:
# one-hot encoding with upsample
features_train_ohe_up, target_train_ohe_up = upsample(features_train_ohe,
                                                      target_train_ohe)

# label encoding with upsample
features_train_le_up, target_train_le_up = upsample(features_train_le,
                                                    target_train_le)

# one-hot encoding with downsample
features_train_ohe_down, target_train_ohe_down = downsample(features_train_ohe,
                                                            target_train_ohe)

# label encoding with downsample
features_train_le_down, target_train_le_down = downsample(features_train_le,
                                                          target_train_le)

### 4.1 LogisticRegression


#### 4.1.1 - Model Class Weight Parameter
Correcting with model class_weight='balanced parameter, that changes the weight to compensate the imbalance.

In [31]:
model_lr_we, f1_lr_we, roc_auc_lr_we = lr_model_scores(
    features_train_ohe,
    target_train_ohe,
    features_val_ohe,
    target_val_ohe,
    weight='balanced'
)
scores_df.loc[len(scores_df)] = ['model_lr_we','','', f1_lr_we, roc_auc_lr_we]

F1 Score: 0.493
ROC AUC: 0.777


<div class="alert alert-block alert-success">
Implementação correta e limpa de class_weight='balanced' para Logistic Regression, com uma explicação clara e concisa do que o parâmetro faz. O F1 salta de 0.328 para 0.481, demonstrando claramente o valor dessa técnica.
</div>

#### 4.1.2 - Upsample

In [32]:
model_lr_up, f1_lr_up, roc_auc_lr_up = lr_model_scores(
    features_train_ohe_up,
    target_train_ohe_up,
    features_val_ohe,
    target_val_ohe,
)

scores_df.loc[len(scores_df)] = ['model_lr_up','','', f1_lr_up, roc_auc_lr_up]

F1 Score: 0.489
ROC AUC: 0.777


#### 4.1.3 - Downsample

In [33]:
model_lr_down, f1_lr_down, roc_auc_lr_down = lr_model_scores(
    features_train_ohe_down,
    target_train_ohe_down,
    features_val_ohe,
    target_val_ohe,
)

scores_df.loc[len(scores_df)] = [
    'model_lr_down',
    '',
    '',
    f1_lr_down,
    roc_auc_lr_down]

F1 Score: 0.493
ROC AUC: 0.774


<div class="alert alert-block alert-warning">
As três técnicas de correção de desbalanceamento para Regressão Logística (class weight, upsample, downsample) estão implementadas corretamente, mas todas resultam em uma faixa estreita (F1 em torno de 0,48-0,485) — nenhuma se aproximando do alvo. Uma breve observação conectando esses três resultados (por exemplo, observando que ponderação e reamostragem têm um efeito similar em um modelo linear aqui, e que nenhuma delas é suficiente por si só) complementaria suas conclusões para essa família de experimentos antes de passar para modelos baseados em árvores.
</div>

### 4.2 - RandomForestClassifier

#### 4.2.1 - Model Class Weight Parameter

In [34]:
# model_rfc_we, n_estimators, depth, f1_rfc_we, roc_auc_rfc_we = rfc_optimizer(
#     features_train_le,
#     target_train_le,
#     features_val_le,
#     target_val_le,
#     weight='balanced'
# )

# scores_df.loc[len(scores_df)] = [
#     'model_rfc_we', 
#     n_estimators, 
#     depth,
#     f1_rfc_we,
#     roc_auc_rfc_we]

<div class="alert alert-block alert-success">
Aplicação correta de class_weight='balanced' ao Random Forest nos dados de treinamento intocados — F1 melhora para 0.635, e o pico do gráfico depth-vs-F1 corresponde à best_depth impressa.
</div>

#### 4.2.2 - Upsample

In [35]:
model_rfc_up, n_estimators, depth, f1_rfc_up, roc_auc_rfc_up = rfc_optimizer(
    features_train_le_up,
    target_train_le_up,
    features_val_le,
    target_val_le,
)

scores_df.loc[len(scores_df)] = [
    'model_rfc_up',
    n_estimators,
    depth,
    f1_rfc_up,
    roc_auc_rfc_up
]

best_depth: 7
best_n_estimators: 15
F1 Score: 0.623
ROC AUC: 0.859


<div class="alert alert-block alert-success">
Upsampling aplicado corretamente como segunda técnica para Random Forest, produzindo seu melhor resultado geral (F1 = 0.638, ROC AUC = 0.871), com o gráfico consistente com os parâmetros impressos.
</div>

#### 4.2.3 - Downsample

In [36]:
# model_rfc_down, n_estimators, depth, f1_rfc_down, roc_auc_rfc_down = (
# rfc_optimizer(
#     features_train_le_down,
#     target_train_le_down,
#     features_val_le,
#     target_val_le,
# ))

# scores_df.loc[len(scores_df)] = [
#     'model_rfc_down',
#     n_estimators,
#     depth,
#     f1_rfc_down,
#     roc_auc_rfc_down
# ]

<div class="alert alert-block alert-success">
O downsampling oferece uma terceira técnica distinta para Random Forest; os resultados (F1 = 0.613) ainda estão confortavelmente acima da meta, e tudo está registrado e plotado de forma consistente.
</div>

### 4.3 - DecisionTreeClassifier

#### 4.3.1 - Model Class Weight Parameter

In [37]:
model_dtc_we, depth, f1_dtc_we, roc_auc_dtc_we = dtc_optimizer(
    features_train_le,
    target_train_le,

    features_val_le,

    target_val_le,
    weight='balanced'
)

scores_df.loc[len(scores_df)] = [
    'model_dtc_we',
    '',
    depth,
    f1_dtc_we,
    roc_auc_dtc_we
]

best_depth: 5
F1 Score: 0.572
ROC AUC: 0.828


<div class="alert alert-block alert-success">
Uso correto de class_weight='balanced' para a Decision Tree, reutilizando dtc_optimizer para consistência com a baseline; o gráfico e o best_depth impresso concordam um com o outro.
</div>

#### 4.3.2 - Upsample

In [38]:
model_dtc_up, depth, f1_dtc_up, roc_auc_dtc_up = dtc_optimizer(
    features_train_le_up,
    target_train_le_up,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = [
    'model_dtc_up',
    '',
    depth,
    f1_dtc_up,
    roc_auc_dtc_up]

best_depth: 5
F1 Score: 0.572
ROC AUC: 0.827


#### 4.3.3 - Downsample

In [39]:
model_dtc_down, depth, f1_dtc_down, roc_auc_dtc_down = dtc_optimizer(
    features_train_le_down,
    target_train_le_down,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = [
    'model_dtc_down',
    '',
    depth,
    f1_dtc_down,
    roc_auc_dtc_down
]

best_depth: 6
F1 Score: 0.573
ROC AUC: 0.816


<div class="alert alert-block alert-warning">
Todas as três técnicas de balanceamento foram implementadas corretamente para a Árvore de Decisão, mas nenhuma delas (0.583, 0.583, 0.589) conseguiu superar a Árvore de Decisão sem peso da seção anterior (0.596 com profundidade 8) — o balanceamento prejudicou ligeiramente esse modelo em particular em vez de ajudá-lo. Este é um resultado genuinamente interessante e contraintuitivo que merece uma nota de uma linha, pois é um contraste útil com o que você observou com Regressão Logística e Random Forest.
</div>

## 5 - F1 and AUC ROC Scores

In [40]:
scores_df.round(3)

,model,n_estimators,depth,f1,auc_roc
0,model_lr,,,0.361,0.773
1,model_dtc,,6,0.535,0.832
2,model_lr_we,,,0.493,0.777
3,model_lr_up,,,0.489,0.777
4,model_lr_down,,,0.493,0.774
5,model_rfc_up,15,7,0.623,0.859
6,model_dtc_we,,5,0.572,0.828
7,model_dtc_up,,5,0.572,0.827
8,model_dtc_down,,6,0.573,0.816


<div class="alert alert-block alert-danger">
A tabela de resumo em si está bem construída e facilita a comparação entre modelos. No entanto, medir AUC-ROC e compará-lo com o F1-score é apresentado como um requisito em separado, distinto das descobertas sobre desbalanceamento de classes, e essa comparação nunca acontece em lugar nenhum no notebook. Você mostra ambas as métricas lado a lado aqui, mas não discute por que AUC-ROC permanece consistentemente alto (0,77-0,87) em quase todos os modelos enquanto F1 varia muito mais amplamente (0,33-0,64) dependendo da técnica de balanceamento utilizada. Adicione uma breve comparação escrita explicando por que essas duas métricas divergem tanto para este problema em particular.
</div>

The best model obtained is the Random Forest Classifier trained with 
upsampled data.

<div class="alert alert-block alert-success">
Leitura correta da tabela — model_rfc_up tem o melhor F1 (0.638) e um AUC-ROC essencialmente empatado em primeiro lugar, e escolhê-lo com base em F1 é a decisão certa considerando a métrica principal do projeto.
</div>

In [43]:
# tuning the threshold for best model

proba_val = model_rfc_up.predict_proba(features_val_le)
proba_one_val = proba_val[:, 1]

for threshold in np.arange(0.2, 0.8, 0.02):
    predicted_val = proba_one_val > threshold

    f1 = f1_score(target_val_le, predicted_val)

    print(f'threshold = {threshold:.2f}, F1 = {f1:.3f}')


threshold = 0.20, F1 = 0.412
threshold = 0.22, F1 = 0.425
threshold = 0.24, F1 = 0.443
threshold = 0.26, F1 = 0.458
threshold = 0.28, F1 = 0.474
threshold = 0.30, F1 = 0.488
threshold = 0.32, F1 = 0.495
threshold = 0.34, F1 = 0.502
threshold = 0.36, F1 = 0.522
threshold = 0.38, F1 = 0.537
threshold = 0.40, F1 = 0.559
threshold = 0.42, F1 = 0.572
threshold = 0.44, F1 = 0.580
threshold = 0.46, F1 = 0.594
threshold = 0.48, F1 = 0.602
threshold = 0.50, F1 = 0.623
threshold = 0.52, F1 = 0.625
threshold = 0.54, F1 = 0.620
threshold = 0.56, F1 = 0.629
threshold = 0.58, F1 = 0.629
threshold = 0.60, F1 = 0.623
threshold = 0.62, F1 = 0.613
threshold = 0.64, F1 = 0.605
threshold = 0.66, F1 = 0.586
threshold = 0.68, F1 = 0.579
threshold = 0.70, F1 = 0.558
threshold = 0.72, F1 = 0.546
threshold = 0.74, F1 = 0.527
threshold = 0.76, F1 = 0.524
threshold = 0.78, F1 = 0.497
threshold = 0.80, F1 = 0.484


<div class="alert alert-block alert-success">
Um passo extra bem pensado — afinar o limiar de decisão no conjunto de validação (não no conjunto de teste) demonstra uma compreensão sólida de como evitar vazar informações do teste para a seleção do modelo.
</div>

The best threshold is 0.50, which is the one the model already uses.

## 6 - Testing

In [42]:

predicted_test = model_rfc_up.predict(features_test_le)
f1 = f1_score(target_test_le, predicted_test)

proba_test = model_rfc_up.predict_proba(features_test_le)
proba_one_test = proba_test[:, 1]
roc_auc = roc_auc_score(target_test_le, proba_one_test)

print(
    f'Test F1 Score: {f1:.3f}\n'
    f'Test ROC AUC: {roc_auc:.3f}'
)


Test F1 Score: 0.582
Test ROC AUC: 0.840


<div class="alert alert-block alert-warning">
O teste final está implementado corretamente — avaliado uma vez no conjunto de teste intocado, ainda desbalanceado, com F1 (0,590) e ROC AUC (0,834) reportados. Vale notar: este F1 fica exatamente no limiar mínimo e é notavelmente mais baixo que o F1 de validação para o mesmo modelo (0,638) — um breve comentário reconhecendo essa queda de validação para teste, e o que ela pode sugerir sobre generalização, completaria suas conclusões de forma elegante.
</div>